In [3]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
import joblib

# Load the data
trainingDataCSV = 'data/winemag-data-130k-v2.csv'
dataFrame = pd.read_csv(trainingDataCSV)
dataFrame.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


In [ ]:
# 1. Keep more columns for a richer model
dataFrame = dataFrame[['variety', 'region_1', 'province', 'winery', 'points', 'price']]

# 2. Drop rows ONLY if the target price is missing
dataFrame = dataFrame.dropna(subset=['price'])

# 3. Apply the Frequency Threshold to group rare items into "Other"
categorical_columns = ['variety', 'region_1', 'province', 'winery']

for col in categorical_columns:
    # Fill missing text with 'Unknown' so the model doesn't crash
    dataFrame[col] = dataFrame[col].fillna('Unknown')
    
    # Keep it if it has 100+ reviews, otherwise label it 'Other'
    counts = dataFrame[col].value_counts()
    dataFrame[col] = dataFrame[col].apply(lambda x: x if counts.get(x, 0) >= 100 else 'Other')
    
    # CRITICAL: Tell pandas this is a category so XGBoost can read it natively
    dataFrame[col] = dataFrame[col].astype('category')

print(f"Rows ready for training: {dataFrame.shape[0]}")
dataFrame.head()

Rows of clean data remaining: 17565


,variety,region_1,points,price
4,Pinot Noir,Willamette Valley,87,65.0
10,Cabernet Sauvignon,Napa Valley,87,19.0
23,Merlot,Paso Robles,87,22.0
25,Pinot Noir,Sonoma Coast,87,69.0
41,Pinot Noir,Willamette Valley,86,22.0
